# HMS merge

按章节合并 Lecture、按 Exercise 合并练习、按时间顺序合并全部考试；最后一个 cell 单独翻译所有合并后的 PDF。

In [1]:
from pathlib import Path
import re
from collections import defaultdict

HMS_DIR = Path(r"E:\OneDrive - MSFT\.master_data\25-26ws\hms")
LECTURE_DIR = HMS_DIR / "Lecture Material"
EXERCISE_DIR = HMS_DIR / "Exercise Material"
EXAM_DIR = HMS_DIR / "Exam Preparation Material"
EXTS = {".ppt", ".pptx", ".pdf"}

def numeric_key(path):
    return tuple(int(x) for x in re.findall(r"\d+", path.stem))

def count_pdf_pages(path):
    from pypdf import PdfReader
    return len(PdfReader(str(path)).pages)

def merge_pdfs(files, output_path):
    from pypdf import PdfWriter
    writer = PdfWriter()
    file_pages = []
    for path in files:
        pages = count_pdf_pages(path)
        file_pages.append((path, pages))
        writer.append(str(path))
    output_path.parent.mkdir(exist_ok=True)
    with output_path.open("wb") as output:
        writer.write(output)
    return file_pages, count_pdf_pages(output_path)

def merge_ppts(files, output_path):
    try:
        import win32com.client as win32
    except ImportError:
        raise ImportError("请先运行：pip install pywin32；该方法需要 Windows + PowerPoint。")
    app = win32.Dispatch("PowerPoint.Application")
    app.Visible = True
    merged = app.Presentations.Add()
    file_pages = []
    try:
        while merged.Slides.Count > 0:
            merged.Slides(1).Delete()
        for path in files:
            src = app.Presentations.Open(str(path.resolve()), ReadOnly=True, WithWindow=False)
            pages = src.Slides.Count
            file_pages.append((path, pages))
            src.Close()
            if pages > 0:
                merged.Slides.InsertFromFile(str(path.resolve()), merged.Slides.Count, 1, pages)
        total_pages = merged.Slides.Count
        merged.SaveAs(str(output_path.resolve()))
    finally:
        merged.Close()
        app.Quit()
    return file_pages, total_pages

def merge_group(files, output_dir, group):
    files = sorted(files, key=numeric_key)
    ppt_files = [p for p in files if p.suffix.lower() in {".ppt", ".pptx"}]
    pdf_files = [p for p in files if p.suffix.lower() == ".pdf"]
    output_dir.mkdir(exist_ok=True)
    if ppt_files:
        pages, total = merge_ppts(ppt_files, output_dir / f"{group}.pptx")
        print(f"{group}.pptx: {len(pages)} files, {total} slides")
    if pdf_files:
        pages, total = merge_pdfs(pdf_files, output_dir / f"{group}.pdf")
        print(f"{group}.pdf: {len(pages)} files, {total} pages")


## 1. Merge lecture material by chapter

In [2]:
groups = defaultdict(list)
for path in LECTURE_DIR.rglob("*"):
    if path.is_file() and path.suffix.lower() in EXTS and not path.name.startswith("~$") and "_merged" not in path.parts:
        match = re.match(r"^HMS_WS2526_Chapter(\d{2})", path.stem, re.IGNORECASE)
        if match:
            groups[f"Chapter{match.group(1)}"].append(path)

for group in sorted(groups, key=lambda name: int(re.search(r"\d+", name).group())):
    print("\n" + "=" * 80)
    print(f"章节 {group}: {len(groups[group])} 个文件")
    merge_group(groups[group], LECTURE_DIR / "_merged", group)



章节 Chapter00: 1 个文件
Chapter00.pdf: 1 files, 20 pages

章节 Chapter01: 1 个文件
Chapter01.pdf: 1 files, 24 pages

章节 Chapter02: 1 个文件
Chapter02.pdf: 1 files, 41 pages

章节 Chapter03: 1 个文件
Chapter03.pdf: 1 files, 49 pages

章节 Chapter04: 1 个文件
Chapter04.pdf: 1 files, 31 pages

章节 Chapter05: 1 个文件
Chapter05.pdf: 1 files, 92 pages

章节 Chapter06: 1 个文件
Chapter06.pdf: 1 files, 157 pages

章节 Chapter07: 1 个文件
Chapter07.pdf: 1 files, 14 pages

章节 Chapter08: 1 个文件
Chapter08.pdf: 1 files, 28 pages

章节 Chapter09: 4 个文件
Chapter09.pdf: 4 files, 181 pages

章节 Chapter10: 1 个文件
Chapter10.pdf: 1 files, 53 pages

章节 Chapter11: 1 个文件
Chapter11.pdf: 1 files, 36 pages

章节 Chapter12: 1 个文件
Chapter12.pdf: 1 files, 16 pages

章节 Chapter13: 1 个文件
Chapter13.pdf: 1 files, 63 pages

章节 Chapter14: 1 个文件
Chapter14.pdf: 1 files, 48 pages

章节 Chapter15: 1 个文件
Chapter15.pdf: 1 files, 11 pages


## 2. Merge exercise material by exercise number

In [3]:
def exercise_group(path):
    relative = path.relative_to(EXERCISE_DIR)
    text = " ".join(relative.parts)
    match = re.search(r"Exercise\s*0*(\d+)", text, re.IGNORECASE)
    if not match:
        match = re.search(r"(?:HMS_|Exercise[_ ])0*(\d+)(?:st|nd|rd|th)?", path.stem, re.IGNORECASE)
    return f"EX{int(match.group(1)):02d}" if match else None

def exercise_sort_key(path):
    name = path.stem.lower()
    role = 0 if any(word in name for word in ("exercise", "task", "setup")) else 1 if "solution" in name else 9
    return role, numeric_key(path), name

groups = defaultdict(list)
for path in EXERCISE_DIR.rglob("*"):
    if path.is_file() and path.suffix.lower() in EXTS and not path.name.startswith("~$") and "_merged" not in path.parts:
        group = exercise_group(path)
        if group:
            groups[group].append(path)

for group in sorted(groups, key=lambda name: int(re.search(r"\d+", name).group())):
    print("\n" + "=" * 80)
    files = sorted(groups[group], key=exercise_sort_key)
    print(f"{group}: {len(files)} 个文件")
    merge_group(files, EXERCISE_DIR / "_merged", group)



EX01: 2 个文件
EX01.pdf: 2 files, 6 pages

EX02: 2 个文件
EX02.pdf: 2 files, 9 pages

EX03: 2 个文件
EX03.pdf: 2 files, 12 pages

EX04: 2 个文件
EX04.pdf: 2 files, 15 pages

EX05: 1 个文件
EX05.pdf: 1 files, 5 pages

EX06: 2 个文件
EX06.pdf: 2 files, 15 pages

EX07: 2 个文件
EX07.pdf: 2 files, 10 pages


## 3. Merge all exam preparation material

In [4]:
def exam_sort_key(path):
    text = path.stem.lower()
    match = re.search(r"(ws|ss)(\d{2,4})(\d{2})?", text)
    if match:
        digits = match.group(2)
        first = int(digits)
        year = first if first >= 1900 else 2000 + (first // 100 if len(digits) == 4 else first)
    else:
        years = [int(x) for x in re.findall(r"(?:19|20)\d{2}", text)]
        year = min(years, default=9999)
    term = 0 if "ss" in text else 1
    solution = 1 if "solution" in text else 0
    return year, term, solution, numeric_key(path), text

exam_files = sorted([
    path for path in EXAM_DIR.rglob("*.pdf")
    if path.is_file() and not path.name.startswith("~$") and "_merged" not in path.parts
], key=exam_sort_key)
if not exam_files:
    raise FileNotFoundError(f"没有找到考试 PDF: {EXAM_DIR}")

exam_output = EXAM_DIR / "_merged" / "HMS_Exams.pdf"
pages, total = merge_pdfs(exam_files, exam_output)
for path, count in pages:
    print(f"{path.name} -> {count} pages")
print(f"HMS_Exams.pdf: {len(pages)} files, {total} pages")


WS1920 Test.pdf -> 32 pages
WS1920 Solution.pdf -> 35 pages
WS1516 Test.pdf -> 27 pages
WS1516 Solution.pdf -> 27 pages
SS16 Test.pdf -> 25 pages
SS16 Solution.pdf -> 26 pages
WS1617 Test.pdf -> 25 pages
WS1617 Solution.pdf -> 27 pages
SS17 Test.pdf -> 29 pages
SS17 Solution_with_Hints.pdf -> 31 pages
WS1718 Test.pdf -> 31 pages
WS1718 Solution_with_Hints.pdf -> 33 pages
SS18 Test.pdf -> 32 pages
SS18 Solution.pdf -> 34 pages
SS18 Solution_with_Hints.pdf -> 35 pages
WS1819 Test.pdf -> 28 pages
WS1819 Solution.pdf -> 28 pages
SS19 Test.pdf -> 33 pages
SS2019 Solution.pdf -> 36 pages
SS2021 Test.pdf -> 34 pages
SS2021 Solution.pdf -> 34 pages
HMS_WS2223_Question.pdf -> 32 pages
HMS_Exams.pdf: 22 files, 674 pages


## 4. Translate all merged PDFs (run separately after merging)

In [ ]:
import shutil
import subprocess
import tempfile

SOURCE_LANG = "en"
TARGET_LANG = "zh"
SERVICE = None
OVERWRITE = False
pdf2zh_cmd = shutil.which("pdf2zh")
if pdf2zh_cmd is None:
    raise RuntimeError("没有找到 pdf2zh，请先运行：pip install pdf2zh")

merged_dirs = [LECTURE_DIR / "_merged", EXERCISE_DIR / "_merged", EXAM_DIR / "_merged"]
pdf_files = sorted(path for folder in merged_dirs for path in folder.glob("*.pdf") if not path.name.startswith("-"))
print(f"待翻译合并文件：{len(pdf_files)} 个")

for pdf_path in pdf_files:
    out_path = pdf_path.with_name("-" + pdf_path.name)
    if out_path.exists() and not OVERWRITE:
        print(f"跳过，已存在：{out_path.name}")
        continue
    print(f"开始翻译：{pdf_path}")
    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)
        tmp_input = tmpdir / pdf_path.name
        shutil.copy2(pdf_path, tmp_input)
        cmd = [pdf2zh_cmd, str(tmp_input), "-li", SOURCE_LANG, "-lo", TARGET_LANG]
        if SERVICE:
            cmd += ["-s", SERVICE]
        result = subprocess.run(cmd, cwd=tmpdir, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print(result.stdout[-2000:])
        if result.returncode != 0:
            print(f"失败：{pdf_path.name}")
            continue
        candidates = list(tmpdir.glob(f"{pdf_path.stem}*dual*.pdf")) or list(tmpdir.glob(f"{pdf_path.stem}*zh*.pdf")) or list(tmpdir.glob(f"{pdf_path.stem}*mono*.pdf"))
        if not candidates:
            print(f"没有找到翻译输出：{pdf_path.name}")
            continue
        if out_path.exists() and OVERWRITE:
            out_path.unlink()
        shutil.move(str(candidates[0]), str(out_path))
        print(f"完成：{out_path.name}")
print("全部翻译处理完成")
